In [1]:
import os 
os.chdir("../")

In [2]:
%pwd

'd:\\Revision\\Covid-19_detection_with_MLOPS'

In [4]:
import dagshub
dagshub.init(repo_owner='shu7620', repo_name='Covid-19_detection_with_MLOPS', mlflow=True)

Accessing as shu7620

Initialized MLflow to track repo "shu7620/Covid-19_detection_with_MLOPS"

Repository shu7620/Covid-19_detection_with_MLOPS initialized!

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    history_path: Path
    plots_dir: Path
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [12]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = self.config.model_evaluation
        
        

        create_directories([eval_config.root_dir, eval_config.plots_dir])

        evaluation_config= EvaluationConfig(
            root_dir=Path(eval_config.root_dir),
            test_data_path=Path(eval_config.test_data_path),
            model_path=Path(eval_config.model_path),
            history_path=Path(eval_config.history_path),
            plots_dir=Path(eval_config.plots_dir),
            mlflow_uri="https://dagshub.com/shu7620/Covid-19_detection_with_MLOPS.mlflow",
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        
        return evaluation_config

In [13]:
import os
import json
import tensorflow as tf
import numpy as np
import mlflow
import mlflow.keras
from pathlib import Path
from dotenv import load_dotenv
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

from cnnClassifier import logger
# from cnnClassifier.entity.config_entity import EvaluationConfig
from cnnClassifier.utils.visualization import generate_training_curves, generate_confusion_matrix_plot

load_dotenv()

class ModelEvaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def evaluate(self):
        """Runs validation calculations, generates assets, and tracking records."""
        # 1. Load data and model
        logger.info("Loading validation data and trained model assets...")
        test_ds = tf.data.Dataset.load(str(self.config.test_data_path))
        model = tf.keras.models.load_model(str(self.config.model_path))

        # 2. Extract predictions for medical matrix calculations
        logger.info("Generating predictions across test dataset tensors...")
        y_true = []
        y_pred_probs = []

        for images, labels in test_ds:
            y_true.extend(np.argmax(labels.numpy(), axis=1))
            preds = model.predict(images, verbose=0)
            y_pred_probs.extend(preds)

        y_true = np.array(y_true)
        y_pred_probs = np.array(y_pred_probs)
        y_pred = np.argmax(y_pred_probs, axis=1)

        # 3. Compute Crucial Medical Metrics
        accuracy = np.mean(y_true == y_pred)
        precision = precision_score(y_true, y_pred, average='binary', pos_label=0) # Assuming 0 is COVID
        recall = recall_score(y_true, y_pred, average='binary', pos_label=0)       # Sensitivity
        f1 = f1_score(y_true, y_pred, average='binary', pos_label=0)
        auc_roc = roc_auc_score(y_true, y_pred_probs[:, 0])

        metrics_dict = {
            "val_accuracy": accuracy,
            "val_precision": precision,
            "val_recall_sensitivity": recall,
            "val_f1_score": f1,
            "val_auc_roc": auc_roc
        }
        
        params_dict = {
            "image_size": str(self.config.params_image_size),  # Convert list to string for MLflow compatibility
            "batch_size": self.config.params_batch_size,
        }

        # 4. Invoke Isolated Plotting Module
        logger.info("Calling internal plotting utilities to isolate diagnostic assets...")
        generate_training_curves(str(self.config.history_path), str(self.config.plots_dir))
        generate_confusion_matrix_plot(y_true, y_pred, classes=["COVID", "Normal"], save_dir=str(self.config.plots_dir))

        # 5. Connect to MLflow remote server via DagsHub
        if self.config.mlflow_uri != "":
            mlflow.set_tracking_uri(self.config.mlflow_uri)
            
        # Ensure experiment exists
        mlflow.set_experiment("COVID19_XRay_Classification")

        with mlflow.start_run():
            logger.info("Logging runtime parameters and diagnostic metrics to MLflow...")
            
            # Log Parameters
            mlflow.log_params(params_dict)
            
            # Log Metrics
            mlflow.log_metrics(metrics_dict)
            
            # Log Plots as Artifacts
            mlflow.log_artifacts(str(self.config.plots_dir), artifact_path="evaluation_plots")
            
            # Log Model Architecture
            mlflow.keras.log_model(model, "model", registered_model_name="CNN_Covid_Detection_Model")
            
        logger.info("Evaluation metrics successfully tracked and artifacts archived.")

In [14]:


STAGE_NAME = "Evaluation Stage"
try:
    logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    model_evaluation = ModelEvaluation(config=eval_config)
    model_evaluation.evaluate()
    logger.info(f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-06-28 19:44:21,733: INFO: 3550827135: >>>>>> stage Evaluation Stage started <<<<<<]
[2026-06-28 19:44:21,745: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-28 19:44:21,747: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-28 19:44:21,748: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-28 19:44:21,749: INFO: common: created directory at: artifacts]
[2026-06-28 19:44:21,749: INFO: common: created directory at: artifacts/model_evaluation]
[2026-06-28 19:44:21,750: INFO: common: created directory at: artifacts/model_evaluation/plots]
[2026-06-28 19:44:21,750: INFO: 1582209500: Loading validation data and trained model assets...]
[2026-06-28 19:44:21,969: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-06-28 19:44:22,776: INFO: 1582209500: Generating predict

2026/06/28 19:44:41 INFO mlflow.tracking.fluent: Experiment with name 'COVID19_XRay_Classification' does not exist. Creating a new experiment.


[2026-06-28 19:44:43,504: INFO: 1582209500: Logging runtime parameters and diagnostic metrics to MLflow...]


2026/06/28 19:44:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/28 19:44:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Revision\Covid-19_detection_with_MLOPS
2026/06/28 19:44:50 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/06/28 19:44:51 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in d:\Revision\Covid-19_detection_with_MLOPS
2026/06/28 19:44:51 INFO mlflow.utils.environment: Detected uv project at d:\Revision\Covid-19_detection_with_MLOPS. Attempting to export requirements via 'uv export'.
2026/06/28 19:44:51 INFO mlflow.utils.uv_utils: Exported 34 dependencies via uv
2026/06/28 19:44:51 INFO mlflow.utils.environment: Successfully exported 34 requirements from uv project. Skipping package capture based inference.
2026/06/28 19:44:56 WARNING mlflow.utils.environment: Failed to resolve installed pip versi

🏃 View run languid-cat-760 at: https://dagshub.com/shu7620/Covid-19_detection_with_MLOPS.mlflow/#/experiments/0/runs/30fa2d93dfe44395bfdab64b4f38f28c
🧪 View experiment at: https://dagshub.com/shu7620/Covid-19_detection_with_MLOPS.mlflow/#/experiments/0
[2026-06-28 19:49:08,884: INFO: 1582209500: Evaluation metrics successfully tracked and artifacts archived.]
[2026-06-28 19:49:08,884: INFO: 3550827135: >>>>>> stage Evaluation Stage completed <<<<<<

x==========x]
